---
title: "Demo van GWDI"
---


## Afleiden van statistiek van de afgelopen 30 jaar


In [ ]:
from pathlib import Path

import pandas as pd
import pastas as ps

from toolbox_continu_inzicht.base.adapters.input.pastas_models import (
    input_pastas_models,
)
from toolbox_continu_inzicht.gwdi.processing.gwdi_reference_stats import (
    compute_df_stats_minima,
    prepare_reference_climate,
)

In [ ]:
# Gebruik lokale voorbeelddata in deze notebook-map.
data_dir = Path.cwd() / "data_sets"

path_precipitation = data_dir / "reference_precipitation_regions.csv"
path_evaporation = data_dir / "reference_evaporation.csv"
path_info = data_dir / "gwdi_pastas_mapping_all.csv"
path_models = data_dir / "pastas_models"

output_path = data_dir / "df_stats_minima.csv"

In [ ]:
# 1) Laad ruwe klimaatinput als dataframes
df_precipitation_regions = pd.read_csv(
    path_precipitation,
    parse_dates=["time"],
    index_col="time",
)
df_evaporation = pd.read_csv(
    path_evaporation,
    parse_dates=["time"],
    index_col="time",
)

# 2) Maak klimaatreeksen in meter
prec100, evap100 = prepare_reference_climate(
    df_precipitation_regions=df_precipitation_regions,
    df_evaporation=df_evaporation,
    precipitation_column="prec_R-R",
    evaporation_column="makkink",
)

# 3) Laad locatie-info en Pastas-modellen
info_locaties = pd.read_csv(path_info)
info_locaties.index = info_locaties["location"].astype(str)
info_locaties["position"] = info_locaties["position"].astype(str)

dict_models_raw = input_pastas_models({"abs_path": path_models})
dict_models_by_location = {
    loc: dict_models_raw[f"{loc}_{info_locaties.loc[loc, 'position']}_tarso"]
    for loc in info_locaties.index
}

# 4) Bereken df_stats_minima
df_stats_minima = compute_df_stats_minima(
    info_locaties=info_locaties,
    dict_models=dict_models_by_location,
    prec100=prec100,
    evap100=evap100,
    ps_module=ps,
)

# Opslaan
df_stats_minima.to_csv(output_path)

df_stats_minima

## Operationele script


In [ ]:
config_path = data_dir / "gwdi_demo.yaml"

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from toolbox_continu_inzicht import Config, DataAdapter
from toolbox_continu_inzicht.gwdi import (
    GwdiInference,
    GwdiKnmiRetrieval,
    GwdiWiwbRetrieval,
)

De configuratie ziet er als volgt uit:

```yaml
GlobalVariables:
    rootdir: "data_sets"
    calc_time: "2025-04-07 08:00:00"

    GwdiWiwbRetrieval:
        lag_days: 0
        publish_days: 35
        target_date: "2025-04-07"
        wiwb_precipitation_source_code: "Knmi.International.Radar.Composite.Combined"

    GwdiKnmiRetrieval:
        lag_days: 0
        publish_days: 35
        target_date: "2025-04-07"

DataAdapter:
    gwdi_input_precipitation_dynamic:
        type: python
    gwdi_input_evaporation_dynamic:
        type: python

    gwdi_input_climate_sampling_locations:
        type: csv
        file: "gwdi_beemster_loc.csv"

    gwdi_input_pastas_mapping:
        type: csv
        file: "gwdi_pastas_mapping.csv"
        dtype:
            location: object
            position: object
    gwdi_input_pastas_models:
        type: pastas_models
        path: "pastas_models"
    gwdi_input_stats_minima:
        type: csv
        file: "df_stats_minima.csv"
        index_col: 0
    gwdi_output:
        type: csv
        file: "gwdi_ouput.csv"

```


In [ ]:
config = Config(config_path=config_path)
config.lees_config()

## Klimaatreeks via WIWB + KNMI

In dit tweede voorbeeld halen we neerslag en verdamping dynamisch op voor dezelfde GWDI-run:

- `GwdiWiwbRetrieval` levert neerslag (`P`) op.
- `GwdiKnmiRetrieval` levert verdamping (`evaporation`) op.

In `gwdi_simple_calculation.yaml` gebruiken beide retrievals dezelfde resample-configuratie (`resample_frequency`, `resample_period_start`, `resample_period_end`) zodat de tijdstappen op elkaar aansluiten.
Daarna voeren we `GwdiInference` uit met deze dynamische invoer.


In [ ]:
required_env = ("WIWB_CLIENT_ID", "WIWB_KEY", "KNMI_API_KEY")
missing_env = [name for name in required_env if not os.getenv(name)]
if missing_env:
    print(
        f"Ontbrekende omgevingsvariabelen voor dynamische GWDI-run: {', '.join(missing_env)}"
    )
else:
    print(f"OK: de benodigde omgevingsvariabelen zijn aanwezig: {required_env}")

In [ ]:
# Gebruik dezelfde YAML-configuratie als in het statische voorbeeld.
data_adapter_dynamic = DataAdapter(config=config)

if missing_env:
    print(
        f"Ontbrekende omgevingsvariabelen voor dynamische GWDI-run: {', '.join(missing_env)}"
    )
else:
    wiwb_module = GwdiWiwbRetrieval(data_adapter=data_adapter_dynamic)
    wiwb_module.run(
        input="gwdi_input_climate_sampling_locations",
        output="gwdi_input_precipitation_dynamic",
    )

    knmi_module = GwdiKnmiRetrieval(data_adapter=data_adapter_dynamic)
    knmi_module.run(
        input="gwdi_input_climate_sampling_locations",
        output="gwdi_input_evaporation_dynamic",
    )

`GwdiInference` vereist exact dezelfde tijdreeksen voor verdamping en neerslag. In de praktijk kan bronbeschikbaarheid per provider toch verschillen. Daarom maken we hieronder expliciet een gezamenlijke (`time`, `fid`)-grid (intersection) voor beide reeksen.


In [ ]:
if missing_env:
    print(
        f"Ontbrekende omgevingsvariabelen voor dynamische GWDI-run: {', '.join(missing_env)}"
    )
else:
    wiwb_grid = wiwb_module.df_out[["time", "fid"]].drop_duplicates()
    knmi_grid = knmi_module.df_out[["time", "fid"]].drop_duplicates()
    common_grid = wiwb_grid.merge(knmi_grid, on=["time", "fid"], how="inner")

    if len(common_grid) == 0:
        raise UserWarning(
            "Geen overlappende (`time`, `fid`)-grid tussen WIWB en KNMI. "
            "Controleer periode/resample-instellingen."
        )

    df_precip_dynamic = wiwb_module.df_out.merge(
        common_grid, on=["time", "fid"], how="inner"
    )
    df_evap_dynamic = knmi_module.df_out.merge(
        common_grid, on=["time", "fid"], how="inner"
    )
    df_precip_dynamic = df_precip_dynamic.sort_values(["time", "fid"]).reset_index(
        drop=True
    )
    df_evap_dynamic = df_evap_dynamic.sort_values(["time", "fid"]).reset_index(
        drop=True
    )

    print("Gezamenlijk grid:")
    display(df_precip_dynamic.merge(df_evap_dynamic, on=["time", "fid"]))

Tenslotte voeren we `GwdiInference` uit met deze dynamische invoer.


In [ ]:
if missing_env:
    print(
        f"Ontbrekende omgevingsvariabelen voor dynamische GWDI-run: {', '.join(missing_env)}"
    )
else:
    data_adapter_dynamic.set_dataframe_adapter(
        "gwdi_input_precipitation_dynamic",
        df_precip_dynamic,
    )
    data_adapter_dynamic.set_dataframe_adapter(
        "gwdi_input_evaporation_dynamic",
        df_evap_dynamic,
    )

    module_dynamic = GwdiInference(data_adapter=data_adapter_dynamic)
    module_dynamic.run(
        input=[
            "gwdi_input_precipitation_dynamic",
            "gwdi_input_evaporation_dynamic",
            "gwdi_input_pastas_mapping",
            "gwdi_input_pastas_models",
            "gwdi_input_stats_minima",
        ],
        output="gwdi_output",
    )

    df_gwdi_dynamic = module_dynamic.df_out.copy()

In [ ]:
module_dynamic.data_adapter.set_global_variable("logging", {"level": "CRITICAL"})
module_dynamic.data_adapter.init_logging(re_initialize=True)
module_dynamic._sync_pastas_logger(module_dynamic.data_adapter.logger)
df_gwdi_input_pastas_models = module_dynamic.data_adapter.input(
    "gwdi_input_pastas_models"
)
info_locaties_plot = module_dynamic.data_adapter.input("gwdi_input_pastas_mapping")
peilbuis_index_mapping = dict(
    zip(
        info_locaties_plot["location"],
        range(len(info_locaties_plot)),
    )
)

index_west = peilbuis_index_mapping["Beemster West D_BITA_F-430"]
index_zuid_a = peilbuis_index_mapping["Beemster Zuid A_KR_F-861"]
index_zuid_b = peilbuis_index_mapping["Beemster Zuid B_BITA_F-407"]

In [ ]:
plt.close("all")
fig, ax = plt.subplots(figsize=(10, 6))
df_gwdi_dynamic["datetime"] = pd.to_datetime(
    df_gwdi_dynamic["datetime"], unit="ms", utc=True
)
df_gwdi_agg = df_gwdi_dynamic.groupby("datetime")["value"].agg(["min", "mean", "max"])
ax.fill_between(
    df_gwdi_agg.index,
    df_gwdi_agg["min"],
    df_gwdi_agg["max"],
    alpha=0.2,
    color="grey",
)
ax.plot(df_gwdi_agg.index, df_gwdi_agg["mean"], color="k")

df_west = df_gwdi_dynamic.query("peilbuisid==@index_west")
ax.plot(df_west["datetime"], df_west["value"], color="#fbce69")

df_zuid_a = df_gwdi_dynamic.query("peilbuisid==@index_zuid_a")
ax.plot(df_zuid_a["datetime"], df_zuid_a["value"], color="#7d93d3")

df_zuid_b = df_gwdi_dynamic.query("peilbuisid==@index_zuid_b")
ax.plot(df_zuid_b["datetime"], df_zuid_b["value"], color="#95cd7a")

ax.set_yscale("log")
ax.grid(True, axis="y")
# ax.set_ylim([0.1, 150])
ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation=45, ha="right");